# Sistema RAG - IA generativa - DecisionCoach AI

**OBJETIVO**

Diseñar e implementar un sistema avanzado de Retrieval-Augmented Generation (RAG) orientado a asistencia profesional, capaz de integrar conocimiento técnico, psicológico y empresarial para mejorar la toma de decisiones y la comunicación.

El sistema busca:

- Indexar y estructurar documentos reales provenientes de múltiples fuentes (IA, psicología, comunicación y negocio)
- Recuperar información relevante mediante búsqueda híbrida (vectorial + léxica)
- Refinar resultados usando técnicas de reranking semántico
- Generar respuestas fundamentadas utilizando modelos de lenguaje (LLMs)
- Reducir alucinaciones mediante validación contra el contexto
- Evaluar automáticamente la calidad de las respuestas (grounding y hallucination score)
- Simular un asistente inteligente (ExecCoach AI) enfocado en decisiones y comunicación profesional

**ENFOQUE DEL SISTEMA**

A diferencia de un RAG tradicional, este sistema incorpora:

- Conocimiento multidisciplinario (IA + psicología + negocio)
- Generación orientada a decisiones y comunicación profesional
- Evaluación automática de confiabilidad
- Arquitectura modular y extensible

📄 Documentos 
- ✂️ Chunking 
- 🔢 Embeddings 
- 🗂️ FAISS 
- 🧠 Query Rewrite (Agente)
- 🔍 Retrieval híbrido 
- 📊 Reranker 
- 🤖 LLM (DecisionCoach AI) 
- 🧹 Validación 
- 📈 Evaluación
  
→ 💬 Respuesta

## 1.Procesos

### 1.1 Librerias

In [106]:
# ## VERSION PARA COLAB

# # =========================================================
# # SETUP LIMPIO Y ESTABLE PARA COLAB (RAG + QWEN)
# # =========================================================

# # 1. Upgrade pip
# !pip install -q --upgrade pip

# # 2. Torch CPU (evita conflictos CUDA con torchvision)
# !pip install -q torch==2.3.0 --index-url https://download.pytorch.org/whl/cpu

# # 3. Transformers + embeddings (versiones compatibles)
# !pip install -q \
#   transformers==4.41.2 \
#   sentence-transformers==2.7.0 \
#   accelerate

# # 4. Vector store + NLP utils
# !pip install -q \
#   faiss-cpu \
#   rank_bm25 \
#   nltk \
#   rouge-score

# # 5. LangChain actualizado
# !pip install -q \
#   langchain \
#   langchain-community \
#   langchain-huggingface

In [1]:
# =========================================================
# SETUP SAGEMAKER (GPU - A10G READY)
# =========================================================

# Upgrade pip
!pip install -q --upgrade pip

# numpy compatible
!pip install -q numpy==1.26.4

# 🔥 TORCH CON CUDA (CLAVE)
!pip uninstall torch -y
!pip uninstall torchvision torchaudio -y

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# transformers + embeddings
!pip install -q transformers==4.41.2 sentence-transformers==2.7.0 accelerate

# FAISS (CPU está bien, GPU opcional)
!conda install -y -c conda-forge faiss-cpu

# utils
!pip install -q rank_bm25 nltk rouge-score

# langchain
!pip install -q langchain langchain-community langchain-huggingface

Found existing installation: torch 2.3.0+cpu
Uninstalling torch-2.3.0+cpu:
  Successfully uninstalled torch-2.3.0+cpu
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.0
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [2]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

### 1.2 Load Dataset

In [75]:
# =========================================================
# 2. LOAD DATASET (EXEC COACH AI + RAG)
# =========================================================

from langchain_community.document_loaders import WebBaseLoader

urls = {
    # =========================
    # 🔹 RAG / IA
    # =========================
    "rag": "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "langchain_rag": "https://docs.langchain.com/oss/python/langchain/rag",
    "models": "https://docs.langchain.com/oss/python/langchain/models",
    "agents": "https://docs.langchain.com/oss/python/langchain/agents",
    "overview": "https://docs.langchain.com/oss/python/langchain/overview",

    # =========================
    # 🧠 DECISION MAKING
    # =========================
    "mental_models": "https://fs.blog/mental-models/",
    "decision_making": "https://en.wikipedia.org/wiki/Decision-making",
    "leader_fall": "https://hbr.org/2026/03/when-senior-leaders-lack-people-skills-transformations-fail",

    # =========================
    # 🧠 PSICOLOGÍA
    # =========================
    "emotional_intelligence": "https://www.psychologytoday.com/us/basics/emotional-intelligence",
    "decision_psychology": "https://www.psychologytoday.com/us/basics/decision-making",
    "cognitive_bias": "https://www.verywellmind.com/cognitive-biases-2794763",

    # =========================
    # 💬 COMUNICACIÓN
    # =========================
    "communication_skills": "https://www.skillsyouneed.com/ips/communication-skills.html",
    "active_listening": "https://www.skillsyouneed.com/ips/active-listening.html",

    # =========================
    # 💼 NEGOCIO / LIDERAZGO
    # =========================
    "leadership_basics": "https://www.ccl.org/articles/leading-effectively-articles/what-is-leadership-a-definition/",
    "decision_business": "https://www.mindtools.com/pages/article/newTED_03.htm",
    "problem_solving": "https://www.mindtools.com/pages/article/newTMC_00.htm",

    # =========================
    # 💰 COMUNICACIÓN REAL
    # =========================
    "elevator_pitch": "https://blog.hubspot.com/sales/elevator-pitch-examples?hubs_content=blog.hubspot.com/sales"
}

# 🔥 Mapeo limpio de categorías (MUY PRO)
category_map = {
    # technical
    "rag": "technical",
    "langchain_rag": "technical",
    "models": "technical",
    "agents": "technical",
    "overview": "technical",

    # decision making
    "mental_models": "decision_making",
    "decision_making": "decision_making",
    "leader_fall": "decision_making",

    # psychology
    "emotional_intelligence": "psychology",
    "decision_psychology": "psychology",
    "cognitive_bias": "psychology",

    # communication
    "communication_skills": "communication",
    "active_listening": "communication",

    # business
    "leadership_basics": "business",
    "decision_business": "business",
    "problem_solving": "business",

    # real world
    "elevator_pitch": "real_world"
}

documents = []

for name, url in urls.items():
    try:
        print(f"🔄 Cargando: {name} -> {url}")

        loader = WebBaseLoader(url)
        docs = loader.load()

        print(f"✅ Listo: {name} | docs: {len(docs)}")

        for doc in docs:
            doc.metadata["source_name"] = name
            doc.metadata["url"] = url
            doc.metadata["category"] = category_map.get(name, "other")

        documents.extend(docs)

    except Exception as e:
        print(f"⚠️ Error cargando {name}: {e}")

🔄 Cargando: rag -> https://en.wikipedia.org/wiki/Retrieval-augmented_generation
✅ Listo: rag | docs: 1
🔄 Cargando: langchain_rag -> https://docs.langchain.com/oss/python/langchain/rag
✅ Listo: langchain_rag | docs: 1
🔄 Cargando: models -> https://docs.langchain.com/oss/python/langchain/models
✅ Listo: models | docs: 1
🔄 Cargando: agents -> https://docs.langchain.com/oss/python/langchain/agents
✅ Listo: agents | docs: 1
🔄 Cargando: overview -> https://docs.langchain.com/oss/python/langchain/overview
✅ Listo: overview | docs: 1
🔄 Cargando: mental_models -> https://fs.blog/mental-models/
✅ Listo: mental_models | docs: 1
🔄 Cargando: decision_making -> https://en.wikipedia.org/wiki/Decision-making
✅ Listo: decision_making | docs: 1
🔄 Cargando: leader_fall -> https://hbr.org/2026/03/when-senior-leaders-lack-people-skills-transformations-fail
✅ Listo: leader_fall | docs: 1
🔄 Cargando: emotional_intelligence -> https://www.psychologytoday.com/us/basics/emotional-intelligence
✅ Listo: emotional

In [76]:
type(documents)

list

### 1.3 Chunking + Overlap

In [77]:
# =========================================================
# 3. CHUNKING + OVERLAP
# =========================================================
# Se divide el texto en fragmentos (chunks) para mejorar la recuperación.
#
# Tradeoff:
# - Chunk pequeño → mayor precisión semántica
# - Chunk grande → mayor contexto
# - Overlap → evita pérdida de información entre chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Tamaño del chunk (balance contexto vs precisión)
    chunk_overlap=100    # Solapamiento para preservar continuidad
)

docs = text_splitter.split_documents(documents)

print(f"Total chunks generados: {len(docs)}")

Total chunks generados: 897


In [78]:
docs[10].page_content

'Process[edit]\nRetrieval-augmented generation (RAG) enhances large language models (LLMs) by incorporating an information-retrieval mechanism that allows models to access and utilize additional data beyond their original training set. Ars Technica notes that "when new information becomes available, rather than having to retrain the model, all that’s needed is to augment the model’s external knowledge base with the updated information" ("augmentation").[4] IBM states that "in the generative phase, the LLM draws from the augmented prompt and its internal representation of its training data to synthesize" an answer.[1]'

### 1.4 Embeddings

In [79]:
# =========================================================
# 4. EMBEDDINGS
# =========================================================
# Se transforman los textos en vectores semánticos para búsqueda eficiente.

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

### 1.5 FAISS (HNSW)

In [80]:
# =========================================================
# 5. FAISS (HNSW)
# =========================================================
# FAISS permite búsqueda vectorial eficiente.
# HNSW (Hierarchical Navigable Small World) optimiza la búsqueda aproximada.

from langchain_community.vectorstores import FAISS
import faiss
import numpy as np

# 1. convertir docs → textos
texts = [doc.page_content for doc in docs]
metadatas = [doc.metadata for doc in docs]

# 2. embeddings
embeddings = embedding_model.embed_documents(texts)
embeddings = np.array(embeddings).astype("float32")
dimension = len(embeddings[0])

# 3. crear índice HNSW
index = faiss.IndexHNSWFlat(dimension, 32)

# 4. agregar embeddings al índice
index.add(embeddings)

# 5. construir vectorstore manualmente
vectorstore = FAISS(
    embedding_model,
    index,
    docstore=None,
    index_to_docstore_id={}
)

# 6. (IMPORTANTE) agregar documentos al docstore
from langchain_community.docstore.in_memory import InMemoryDocstore
import uuid

docstore = InMemoryDocstore({})
index_to_docstore_id = {}

for i, text in enumerate(texts):
    doc_id = str(uuid.uuid4())
    docstore._dict[doc_id] = docs[i]
    index_to_docstore_id[i] = doc_id

vectorstore.docstore = docstore
vectorstore.index_to_docstore_id = index_to_docstore_id


retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 20}
)

In [81]:
vectorstore

### 1.6 Retrieval (Hybrid: Faiss + BM25)

In [82]:
# =========================================================
# 6. RETRIEVAL (HYBRID SEARCH)
# =========================================================
# Se combina:
# - FAISS → búsqueda semántica
# - BM25 → búsqueda léxica

from rank_bm25 import BM25Okapi

tokenized_docs = [doc.page_content.lower().split() for doc in docs]
bm25 = BM25Okapi(tokenized_docs)

def hybrid_retrieval(query, k=5):
    # FAISS
    faiss_docs = retriever.invoke(query)

    # BM25
    scores = bm25.get_scores(query.split())
    # scores = bm25.get_scores(query.lower().split())
    top_k = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    bm25_docs = [docs[i] for i in top_k]

    # Combinar resultados
    combined = faiss_docs + bm25_docs
    return combined

### 1.7 Reranker (Cross-Encoder)

In [83]:
# =========================================================
# 7. RERANKER
# =========================================================
# El reranker mejora la calidad del retrieval evaluando pares (query, doc)

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_documents(query, documents, threshold=0.2):
    pairs = [[query, doc.page_content] for doc in documents]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)

    filtered = [doc for doc, score in ranked if score > threshold]

    return filtered[:2]

### 1.8 LLM's

#### 1.8.1 LLM QWEN

In [104]:
import torch
print(torch.cuda.is_available())

True


In [85]:
# =========================================================
# 8.1 LLM (QWEN) - VERSION ESTABLE 
# =========================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "Qwen/Qwen1.5-1.8B-Chat"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Modelo (SIN bitsandbytes, SIN CUDA errors)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=250,
    temperature=0.3,
    do_sample=True,               #  importante
    top_p=0.9,                    #  mejora fluidez
    top_k=50,                     #  control fino
    repetition_penalty=1.1,       # evita loops
    eos_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

# LLM para LangChain
llm = HuggingFacePipeline(pipeline=pipe)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


#### 1.8.2 Flan T5 Large

In [86]:
# =========================================================
# 8.2 Flant T5 LARGE 
# =========================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

small_model_name = "google/flan-t5-large"

small_tokenizer = AutoTokenizer.from_pretrained(small_model_name)
small_model = AutoModelForSeq2SeqLM.from_pretrained(small_model_name)

small_pipe = pipeline(
    "text2text-generation",
    model=small_model,
    tokenizer=small_tokenizer,
    max_new_tokens=50,
    temperature=0.0,   #  importante
    do_sample=False    #  más determinista
)

llm_small = HuggingFacePipeline(pipeline=small_pipe)

### 1.9 Citaion per sentence

In [87]:
# =========================================================
# 9. CITATION PER SENTENCE
# =========================================================
# Se añade trazabilidad a cada oración generada.

def add_citations(answer, docs):
    sentences = answer.split(".")

    cited_sentences = []

    for i, sentence in enumerate(sentences):
        sentence = sentence.strip()
        if not sentence:
            continue

        doc = docs[i % len(docs)]
        source_name = doc.metadata.get("source_name", "unknown")

        cited_sentences.append(f"{sentence} [{source_name}]")

    return ". ".join(cited_sentences)

### 1.10 Halucination Guard

In [88]:
# =========================================================
# 10. HALLUCINATION GUARD
# =========================================================
# Detecta si la respuesta contiene información fuera del contexto.

def hallucination_guard(answer, context):
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())

    # 🔥 PROTECCIÓN CONTRA RESPUESTA VACÍA
    if len(answer_words) == 0:
        return "⚠️ Respuesta vacía (posible fallo del LLM)", 0.0

    overlap = len(answer_words & context_words)
    score = overlap / len(answer_words)

    if score < 0.3:
        return "⚠️ Posible alucinación detectada", score
    else:
        return "✅ Respuesta confiable", score

### 1.11 Grounding Evaluation

In [89]:
# =========================================================
# 11. GROUNDING EVALUATION
# =========================================================
# Mide qué tan basada está la respuesta en el contexto.

def grounding_score(answer, context):
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())

    if len(answer_words) == 0:
        return 0.0

    overlap = len(answer_words & context_words)

    # 🔥 más tolerante a paráfrasis
    return overlap / (len(answer_words) * 0.7)

### 1.12 Bleu / Rouge

In [90]:
# =========================================================
# 12. BLEU / ROUGE
# =========================================================

from rouge_score import rouge_scorer

def evaluate_rouge(reference, generated):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    return scores

## 2.Pipeline Integrado

### 2.0 Función Soporte

In [91]:
def build_context(docs, max_chars=1000):
    context = ""

    for doc in docs:
        chunk = doc.page_content.strip()

        if len(context) + len(chunk) > max_chars:
            break

        context += chunk + "\n\n"

    return context.strip()

In [92]:
import re

def clean_answer(text):

    # eliminar bloques de código
    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL)

    # eliminar citation needed
    text = re.sub(r"\[citation needed\]", "", text, flags=re.IGNORECASE)

    # eliminar sources
    text = re.sub(r"\[source\s*\d+\]", "", text)

    text = re.sub(r"\[.*?needed\]", "", text, flags=re.IGNORECASE)

    # limpiar espacios
    text = re.sub(r"\s+", " ", text)

    return text.strip()


In [93]:
def safe_generate(prompt, query):
    answer = llm.invoke(prompt)

    if not answer or len(answer.strip()) == 0:
        return f"I could not find a clear answer in the context for: {query}"

    return answer.strip()

In [94]:
def unique_sources(docs):
    seen = set()
    unique = []

    for doc in docs:
        key = doc.metadata.get("source_name")

        if key not in seen:
            seen.add(key)
            unique.append(doc)

    return unique

In [95]:
def diversify_sources(docs, max_docs=6):
    seen = set()
    diversified = []

    for doc in docs:
        source = doc.metadata.get("source_name")

        if source not in seen:
            diversified.append(doc)
            seen.add(source)

        if len(diversified) >= max_docs:
            break

    return diversified

In [96]:
def enforce_context(answer, context):
    sentences = answer.split(".")
    valid_sentences = []

    for s in sentences:
        s = s.strip()
        if not s:
            continue

        # si la mayoría de palabras están en el contexto → ok
        overlap = len(set(s.lower().split()) & set(context.lower().split()))

        if overlap / max(len(s.split()), 1) > 0.4:
            valid_sentences.append(s)

    return ". ".join(valid_sentences)

In [97]:
import re

def fix_basic_typos(query):
    # arregla palabras pegadas con mayúsculas raras
    query = re.sub(r'\b([a-z]+)[A-Z]\b', r'\1', query)

    # casos específicos comunes
    query = query.replace("isS", "is")

    return query

In [98]:
def normalize_query(query):
    return query.strip().lower()

### 2.1 Agente Rewrite Query

In [99]:
# =========================================================
# AGENTE REWRITE QUERY
# =========================================================

def rewrite_query_llm(query):
    prompt = f"""
    You are a text correction tool.
    
    Your ONLY task is to fix spelling errors.
    
    STRICT RULES:
    - Return EXACTLY the same sentence, correcting ONLY spelling mistakes
    - DO NOT answer the question
    - DO NOT explain anything
    - DO NOT rephrase
    - DO NOT change grammar or wording
    - DO NOT add or remove words
    - DO NOT change word order
    - If the sentence is already correct, return it EXACTLY as is
    
    EXAMPLES:
    Input: What iss RAG?
    Output: What is RAG?
    
    Input: Why are mental models important?
    Output: Why are mental models important?
    
    Input: How does RAG improve LLM?
    Output: How does RAG improve LLM?
    
    Sentence:
    {query}
    
    Output:
    """
    return llm_small.invoke(prompt).strip().split("\n")[0]

In [100]:
rewrite_query_llm("Why are mentall models important?")

'Why are mental models important?'

### 2.2 Pipeline Principal: Agente DecisionCoachAI

In [116]:
# =========================================================
# Pipeline Principal: Agente DecisionCoachAI
# =========================================================

def rag_pipeline(query, reference_answer=None):

    # 0.Rewrite Query
    query = rewrite_query_llm(query)

    query = fix_basic_typos(query)

    # 🔥 NUEVO: normalización consistente
    query = normalize_query(query)

    # 1. Retrieval híbrido
    retrieved_docs = hybrid_retrieval(query)

    # 2. Reranking
    reranked_docs = rerank_documents(query, retrieved_docs)

    # top_docs = reranked_docs[:4]
    top_docs = diversify_sources(reranked_docs, max_docs=6)

    # 3. Construcción de contexto
    # context = "\n\n".join([doc.page_content for doc in top_docs])
    context = build_context(top_docs)
    
    print("\n" + "="*50)
    print("🔍 DEBUG RETRIEVAL")
    print(f"Query: {query}")
    print(f"Retrieved docs: {len(retrieved_docs)}")
    print(f"Reranked docs: {len(reranked_docs)}")
    print(f"Top docs: {len(top_docs)}")
    print("="*50)
    
    # 4. Prompt
    prompt = f"""
    You are DecisionCoach AI, a professional assistant for decision-making and workplace communication.
    
    Your goal is to help users think clearly, make better decisions, and communicate effectively in professional contexts.
    
    RULES:
    - Base your answer primarily on the context, but you may use general reasoning if needed
    - Do NOT invent specific facts not supported by the context
    - Provide useful, practical, and actionable guidance
    - Do NOT return empty answers
    - If the context is weak, still provide a reasonable structured answer
    - If nothing relevant is found at all, say: "Not found in context"
    
    RESPONSE STRUCTURE:
    1. Understanding → briefly restate the situation
    2. Analysis → explain the situation using reasoning
    3. Recommendation → give a clear actionable suggestion
    4. Example → include a practical example if communication is involved
    
    TONE:
    - Professional and clear
    - Structured and concise (max 4–6 sentences)
    - Not overly emotional
    
    Context:
    {context}
    
    Question:
    {query}
    
    Answer:
    """

    # 5. Generación
    # answer = llm.invoke(prompt)
    answer = safe_generate(prompt, query)

    answer = clean_answer(answer)

    answer = enforce_context(answer, context)

    # 6. Citations
    answer_with_citations = add_citations(answer, top_docs)

    # 7. Evaluación
    hallucination_msg, h_score = hallucination_guard(answer, context)
    g_score = grounding_score(answer, context)

    rouge_scores = None
    if reference_answer:
        rouge_scores = evaluate_rouge(reference_answer, answer)
    
    unique_docs = unique_sources(top_docs)
    return {
      "answer": answer,
      "answer_with_citations": answer_with_citations,
      "sources": unique_docs,
      "hallucination": hallucination_msg,
      "hallucination_score": h_score,
      "grounding_score": g_score,
      "rouge": rouge_scores
    }

### 2.3 Test

In [112]:
# Test

questions = [
    # =========================
    # 🔹 RAG / IA
    # =========================
    "What iss RAG?",
    "How does RAG improvee LLM?",
    "What role do embeddings play in RAG systems?",

    # =========================
    # 💼 EXEC COACH AI (DECISION + COMUNICACIÓN)
    # =========================
    "What aare mental models?",
    "Why are mental models important?",
    "What is emotional intelligence?",
    "What is leadership?"
]

In [113]:
for q in questions:
    fixed = rewrite_query_llm(q)

    print("\n" + "="*60)
    print("🧠 ORIGINAL :", q)
    print("🛠️ FIXED    :", fixed)


🧠 ORIGINAL : What iss RAG?
🛠️ FIXED    : What is RAG?

🧠 ORIGINAL : How does RAG improvee LLM?
🛠️ FIXED    : How does RAG improve LLM?

🧠 ORIGINAL : What role do embeddings play in RAG systems?
🛠️ FIXED    : What role do embeddings play in RAG systems?

🧠 ORIGINAL : What aare mental models?
🛠️ FIXED    : What are mental models?

🧠 ORIGINAL : Why are mental models important?
🛠️ FIXED    : Why are mental models important?

🧠 ORIGINAL : What is emotional intelligence?
🛠️ FIXED    : What is emotional intelligence?

🧠 ORIGINAL : What is leadership?
🛠️ FIXED    : What is leadership?


In [114]:
for q in questions:
    result = rag_pipeline(q)

    print("\n" + "="*60)
    print(f"🧠 QUESTION: {q}")
    print("="*60)

    print("\n📌 Answer:")
    print(result["answer_with_citations"])

    print("\n📊 Evaluation:")
    print(f"• Hallucination: {result['hallucination']}")
    print(f"• Hallucination Score: {result['hallucination_score']:.2f}")
    print(f"• Grounding Score: {result['grounding_score']:.2f}")

    print("\n📚 Sources:")
    for i, doc in enumerate(result["sources"]):
        print(f"\n[{i+1}] {doc.metadata['source_name'].upper()}")
        print(f"🔗 {doc.metadata['url']}")


🔍 DEBUG RETRIEVAL
Query: what is rag?
Retrieved docs: 25
Reranked docs: 2
Top docs: 1

🧠 QUESTION: What iss RAG?

📌 Answer:
Rag is a retrieval-augmented generation technique that leverages large language models (LLMs) to retrieve and incorporate external data sources into their responses [rag]. In this approach, LLMs first reference a predefined set of documents, which serve as the foundation for generating responses [rag]. The retrieved information is then integrated with the LLM's existing training data, allowing it to leverage domain-specific knowledge and up-to-date information [rag]. This process can be particularly beneficial in scenarios where the LLM's training data is limited or outdated, enabling it to generate more accurate and relevant responses to user queries [rag]. Enhanced Information Retrieval: By referring to external data sources, LLMs can access a broader range of information that might not have been included in their training data [rag]

📊 Evaluation:
• Hallucinat

In [129]:
qa_pairs = [
    {
        "question": "What iss RAG?",
        "reference": "RAG is a technique that allows large language models to retrieve and incorporate information from external data sources. In this approach, the model first refers to a set of documents and then generates responses based on them. These documents complement the model's existing training data, allowing it to use more up-to-date and domain-specific information. This helps models generate more accurate responses and access information from sources such as internal data or authoritative content."
    },
    {
        "question": "How does RAG improvee LLM?",
        "reference": "RAG improves large language models by retrieving relevant information before generating responses, instead of relying only on static training data. It pulls data from sources such as databases, documents, or the web, allowing models to use updated and more accurate information. By combining retrieval with generation, RAG helps models stay grounded in facts and reduces hallucinations."
    },
    {
        "question": "What role do embeddings play in RAG systems?",
        "reference": "In RAG systems, embeddings are used to convert data into numerical vector representations that capture meaning. These vectors are stored in a database and allow the system to efficiently retrieve the most relevant documents based on similarity to the user query."
    },
    {
        "question": "What aare mental models?",
        "reference": "Mental models are simplified explanations of how things work that help us understand and navigate the world. They focus on key information while ignoring unnecessary details, allowing us to simplify complexity and make better decisions."
    },
    {
        "question": "Why are mental models important?",
        "reference": "Mental models highlight key information while ignoring irrelevant details and are tools for compressing complexity into manageable chunks. They help us understand the world."
    },
    {
        "question": "What is emotional intelligence?",
        "reference": "EEmotional intelligence refers to the ability to identify and manage one’s own emotions, as well as the emotions of others. It includes emotional awareness, which is the ability to identify and name one’s emotions, the ability to use emotions in tasks like thinking and problem solving, and the ability to regulate emotions in oneself and others."
    },
    {
        "question": "What is leadership?",
        "reference": "Leadership is a social process in which people work together to achieve results that they could not accomplish alone. It is not only about individual roles or skills, but about collective effort within an organization, where everyone contributes to achieving shared goals."
    }
]

In [130]:
for qa in qa_pairs:
    result = rag_pipeline(
        qa["question"],
        reference_answer=qa["reference"]
    )

    print("\n" + "="*60)
    print(f"🧠 QUESTION: {qa['question']}")
    print("="*60)

    print("\n📌 Answer:")
    print(result["answer_with_citations"])

    print("\n📊 Evaluation:")
    print(f"• Hallucination: {result['hallucination']}")
    print(f"• Hallucination Score: {result['hallucination_score']:.2f}")
    print(f"• Grounding Score: {result['grounding_score']:.2f}")

    # ROUGE
    if result["rouge"]:
        print(f"• ROUGE-1: {result['rouge']['rouge1'].fmeasure:.2f}")
        print(f"• ROUGE-L: {result['rouge']['rougeL'].fmeasure:.2f}")

    print("\n📚 Sources:")
    for i, doc in enumerate(result["sources"]):
        print(f"\n[{i+1}] {doc.metadata['source_name'].upper()}")
        print(f"🔗 {doc.metadata['url']}")


🔍 DEBUG RETRIEVAL
Query: what is rag?
Retrieved docs: 25
Reranked docs: 2
Top docs: 1

🧠 QUESTION: What iss RAG?

📌 Answer:
Rag is a type of information retrieval technique where large language models (LLMs) leverage external data sources to augment their training data [rag]. This process allows the LLM to generate more accurate and relevant responses to user queries, even when the original training data does not cover all aspects of the topic [rag]

📊 Evaluation:
• Hallucination: ✅ Respuesta confiable
• Hallucination Score: 0.52
• Grounding Score: 0.75
• ROUGE-1: 0.44
• ROUGE-L: 0.31

📚 Sources:

[1] RAG
🔗 https://en.wikipedia.org/wiki/Retrieval-augmented_generation

🔍 DEBUG RETRIEVAL
Query: how does rag improve llm?
Retrieved docs: 25
Reranked docs: 2
Top docs: 1

🧠 QUESTION: How does RAG improvee LLM?

📌 Answer:
Incorporating Information Retrieval: RAG uses information retrieval techniques to extract relevant text from various sources, such as databases, uploaded documents, or web 

---

FIN

---